#Biliotheken, Funktionen

In [ ]:
import numpy as np
import pandas as pd
import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Datenladen

In [ ]:
df_calls = pd.read_pickle('Cl_Calls.pkl')
df_cont = pd.read_pickle('Cl_Cont.pkl')
df_spend = pd.read_pickle('Cl_Spend.pkl')
df_deals = pd.read_pickle('Cl_Deals.pkl')

#Zeitreihenanalyse:
1. Analysieren Sie den Trend der Deal-Erstellungen im Zeitverlauf und deren
Zusammenhang mit Anrufen.
2. Untersuchen Sie die Verteilung der Abschlusszeiten und die Dauer vom Erstellen bis
zum Abschluss eines Deals.

##1. Analysieren Sie den Trend der Deal-Erstellungen im Zeitverlauf und deren Zusammenhang mit Anrufen.

In [ ]:
df_deals.head(2)

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,SLA Seconds,Deutsch_Level_Clean
0,5805028000056864695,Ben Hall,NaT,Unknown,New Lead,Unknown,/eng/test,03.07.23women,v16,women,...,Unknown,2024-06-21 15:30:00,NaN,NaN,NaN,NaN,5805028000056849495,Unknown,NaN,Unknown
1,5805028000056859489,Ulysses Adams,NaT,Unknown,New Lead,Unknown,/at-eng,Unknown,Unknown,Unknown,...,Morning,2024-06-21 15:23:00,6.0,NaN,NaN,2000.0,5805028000056834471,Unknown,NaN,Unknown


In [ ]:
df_deals['Deal Date'] = df_deals['Created Time'].dt.date
df_deals['Deal Time'] = df_deals['Created Time'].dt.time

In [ ]:
df_calls.head(2)

,Id,Call Start Time,Call Owner Name,CONTACTID,Call Type,Call Duration (in seconds),Call Status,Outgoing Call Status,Scheduled in CRM
0,5805028000000805001,2023-06-30 08:43:00,John Doe,NaN,Inbound,171.0,Received,NaN,NaN
1,5805028000000768006,2023-06-30 08:46:00,John Doe,NaN,Outbound,28.0,Attended Dialled,Completed,0.0


In [ ]:
df_calls['Call Date'] = df_calls['Call Start Time'].dt.date
df_calls['Call Time'] = df_calls['Call Start Time'].dt.time

In [ ]:
deals_day = df_deals.groupby('Deal Date').size().reset_index(name='Deals Count')
calls_day = df_calls.groupby('Call Date').size().reset_index(name='Calls Count')

In [ ]:
date_df = pd.merge(deals_day, calls_day, left_on='Deal Date',
                   right_on='Call Date', how='outer').dropna()

In [ ]:
date_df.head()

,Deal Date,Deals Count,Call Date,Calls Count
41,2023-07-07,10.0,2023-07-07,27.0
42,2023-07-08,10.0,2023-07-08,77.0
43,2023-07-09,10.0,2023-07-09,153.0
44,2023-07-10,14.0,2023-07-10,100.0
45,2023-07-11,6.0,2023-07-11,259.0


In [ ]:
fig = go.Figure()
date_df['Date'] = pd.to_datetime(date_df['Call Date'])

fig.add_trace(
    go.Scatter(x=date_df['Call Date'], y=date_df['Calls Count'],
               name='Calls')
)

call_trend = px.scatter(date_df, x='Date', y='Calls Count',
                        trendline='ols', template='plotly_white')
trend_data = call_trend.data[1]
fig.add_trace(trend_data)

fig.add_trace(
    go.Scatter(x=date_df['Date'], y=date_df['Deals Count'],
               name='Deals',
               )
)

deal_trend = px.scatter(date_df, x='Date', y='Deals Count',
                        trendline='ols', template='plotly_white')
trend_data = deal_trend.data[1]
fig.add_trace(trend_data)

fig.update_layout(
    title="Calls & Deals over Time",
    title_x=0.5,
    xaxis_title="Дата",
    yaxis_title="Количество",
    hovermode='x unified',
    height=600,
    plot_bgcolor='gainsboro'
)

fig.show()

Aus dem Diagramm geht hervor, dass die Häufigkeit der Anrufe nicht unbedingt mit der Häufigkeit der Geschäftsabschlüsse korreliert.

##2. Untersuchen Sie die Verteilung der Abschlusszeiten und die Dauer vom Erstellen bis zum Abschluss eines Deals.

In [ ]:
df_deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,SLA Seconds,Deutsch_Level_Clean,Deal Date,Deal Time
0,5805028000056864695,Ben Hall,NaT,Unknown,New Lead,Unknown,/eng/test,03.07.23women,v16,women,...,NaN,NaN,NaN,NaN,5805028000056849495,Unknown,NaN,Unknown,2024-06-21,15:30:00
1,5805028000056859489,Ulysses Adams,NaT,Unknown,New Lead,Unknown,/at-eng,Unknown,Unknown,Unknown,...,6.0,NaN,NaN,2000.0,5805028000056834471,Unknown,NaN,Unknown,2024-06-21,15:23:00
2,5805028000056832357,Ulysses Adams,2024-06-21 14:45:00,D - Non Target,Lost,Non target,/at-eng,engwien_AT,b1-at,21_06_2024,...,NaN,NaN,NaN,NaN,5805028000056854421,Unknown,1603.0,Unknown,2024-06-21,00:00:00
3,5805028000056824246,Eva Kent,2024-06-21 13:32:00,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,bloggersvideo14com,recentlymoved,...,NaN,NaN,NaN,NaN,5805028000056889351,Unknown,3604.0,Unknown,2024-06-21,00:00:00
4,5805028000056873292,Ben Hall,2024-06-21 13:21:00,D - Non Target,Lost,Non target,/eng,discovery_DE,website,Unknown,...,NaN,NaN,NaN,NaN,5805028000056876176,Unknown,3192.0,Unknown,2024-06-21,00:00:00


In [ ]:
df_deals['Close Date'] = df_deals['Closing Date'].dt.date
df_deals['Close Time'] = df_deals['Closing Date'].dt.time

In [ ]:
df_deals[['Close Date', 'Close Time', 'Deal Date', 'Deal Time']]

,Close Date,Close Time,Deal Date,Deal Time
0,NaT,NaT,2024-06-21,15:30:00
1,NaT,NaT,2024-06-21,15:23:00
2,2024-06-21,14:45:00,2024-06-21,00:00:00
3,2024-06-21,13:32:00,2024-06-21,00:00:00
4,2024-06-21,13:21:00,2024-06-21,00:00:00
...,...,...,...,...
21587,2023-07-06,00:00:00,2023-07-04,09:34:00
21589,2023-08-29,00:00:00,2023-07-04,07:10:00
21590,2023-08-29,00:00:00,2023-07-03,20:39:00
21591,2023-07-09,00:00:00,2023-07-03,20:17:00


In [ ]:
df_calls['Month'] = df_calls['Call Start Time'].dt.to_period('M').dt.to_timestamp()
df_deals['Month'] = df_deals['Closing Date'].dt.to_period('M').dt.to_timestamp()

monthly = df_deals.groupby('Month').agg(
        deals_count=('Id','count'),
        ).reset_index().sort_values('Month')

mean_all = monthly['deals_count'].mean()
median_all = monthly['deals_count'].median()

In [ ]:
monthly = df_deals.groupby(
    'Month').agg(
        deals_count=('Id','count'),
        ).reset_index().sort_values('Month')

mean_all = monthly['deals_count'].mean()
median_all = monthly['deals_count'].median()

monthly['ma3'] = monthly['deals_count'].rolling(3, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=monthly['Month'],
        y=monthly['deals_count'],
        mode='lines+markers',
        name='deals_count'
        )
    )

fig.add_trace(
    go.Scatter(
        x=monthly['Month'],
        y=monthly['ma3'],
        mode='lines',
        name='MA(3)',
        line=dict(dash='dash', color='red')
        )
    )

fig.update_layout(
    title='Monthly deals by Month',
    xaxis_title='Month',
    yaxis_title='Deals count'
    )

fig.show()

тут видно, что пик закрытіх сделок приходится на апрель 2024 а сентябре замечен резкий спад, пока непонятно с чем он связан

In [ ]:
df_filtered = df_deals[df_deals['Product'] != 'Unknown'].copy()

agg = df_filtered.groupby(['Month','Product']).size().reset_index(name='Count Deals')

fig = px.line(
    agg, x='Month',
    y='Count Deals',
    color='Product',
    facet_col='Product',
    facet_col_wrap=3,
    title='Monthly Deals by Product'
    )

fig.update_layout(height=500)
fig.show()

Die Aufschlüsselung nach Produkten lässt erkennen, dass Web Developer aus bestimmten Gründen später als die anderen an Fahrt gewonnen hat. Außerdem wird sofort deutlich, dass UX/UI Design das beliebteste Produkt ist.

In [ ]:
df_deals['Duration days'] = (df_deals['Closing Date'] - df_deals['Created Time']).dt.days
df_deals['Duration hours'] = (df_deals['Closing Date'] - df_deals['Created Time']).dt.total_seconds() / 3600

In [ ]:
monthly_dur = (df_deals.assign(Month=df_deals['Closing Date'].dt.to_period('M').dt.to_timestamp())
               .groupby('Month')
               .agg(deals_count=('Id','count'), duration_mean=('Duration days','mean'))
               .sort_index())

all_months = pd.date_range(
    monthly_dur.index.min(),
    monthly_dur.index.max(),
    freq='MS')
monthly_dur = monthly_dur.reindex(all_months) \
                 .rename_axis('Month') \
                 .reset_index()
monthly_dur['deals_count'] = monthly_dur['deals_count'].fillna(0).astype(int) # Corrected 'Count Deals' to 'deals_count'

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(
        x=monthly_dur['Month'],
        y=monthly_dur['duration_mean'],
        name='Avg duration (days)',
        marker_color='steelblue',
        opacity=0.8
        ),
    secondary_y=False
    )

fig.add_trace(
    go.Scatter(
        x=monthly_dur['Month'],
        y=monthly_dur['deals_count'],
        mode='lines+markers',
        name='Closed deals (count)',
        line=dict(color='crimson', width=2)
        ),
    secondary_y=True
    )

fig.update_xaxes(tickformat='%b %Y', tickangle=45)
fig.update_yaxes(title_text='Avg duration (days)', secondary_y=False)
fig.update_yaxes(title_text='Closed deals count', secondary_y=True)
fig.update_layout(
    title='Average duration of closed deals (bars) + number of closed deals (line)',
    bargap=0.12,
    height=400
    )

fig.show()

 In einem bestimmten Monat gab es nur sehr wenige Transaktionen (1–2) und diese hatten eine enorme Laufzeit → hoher Durchschnitt.

In [ ]:
cutoff = pd.Timestamp('2024-07-01')
monthly_dur_filtered = monthly_dur.loc[monthly_dur['Month'] <= cutoff].copy()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=monthly_dur_filtered['Month'],
        y=monthly_dur_filtered['duration_mean'],
        name='Avg duration (days)',
        marker_color='steelblue',
        opacity=0.8
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=monthly_dur_filtered['Month'],
        y=monthly_dur_filtered['deals_count'],
        mode='lines+markers',
        name='Closed deals (count)',
        line=dict(color='crimson', width=2)
    ),
    secondary_y=True
)

fig.update_xaxes(tickformat='%b %Y', tickangle=45)
fig.update_yaxes(title_text='Avg duration (days)', secondary_y=False)
fig.update_yaxes(title_text='Closed deals count', secondary_y=True)
fig.update_layout(
    title='Average duration of closed deals (bars) + number of closed deals (line) — up to Jul 2024',
    bargap=0.12,
    height=400
)

fig.show()

In [ ]:
df_own = df_deals[df_deals['Deal Owner Name'] != 'Unknown']

In [ ]:
grouped = (
    df_own
    .groupby(["Deal Owner Name", "Month"])
    .size()
    .reset_index(name="Deals Count")
)

pivot_table = grouped.pivot_table(
    index="Deal Owner Name",
    columns="Month",
    values="Deals Count",
)

total_per_manager = pivot_table.sum(axis=1)
avg_per_year = total_per_manager / 12
low_performers = avg_per_year[avg_per_year < 5].index.tolist()
y_labels = [
    f"<b style='color:crimson'>{name}</b>" if name in low_performers else name
    for name in pivot_table.index
]

fig = px.imshow(
    pivot_table,
    text_auto=True,
    color_continuous_scale='matter',
    labels=dict(y='Owners', color="Count Deals"),
    title="Closed Deals by Owners"
)

fig.update_xaxes(side="top")
fig.update_layout(
    height=800,
    showlegend=True,
    plot_bgcolor='ivory'
)

fig.update_yaxes(
    ticktext=y_labels,
    tickvals=np.arange(len(y_labels))
)

fig.show()

In [ ]:
grouped = (
    df_own
    .groupby(['Deal Owner Name', 'Month'])
    .agg(
        total_deals=('Id', 'count'),
        clients=('Stage', lambda x: (x == 'Payment Done').sum())
    )
    .reset_index()
)

grouped['C1 (%)'] = grouped['clients'] / grouped['total_deals'] * 100

pivot_conv = grouped.pivot_table(
    index='Deal Owner Name',
    columns='Month',
    values='C1 (%)'
)

avg_conv_per_owner = pivot_conv.mean(axis=1)
high_perf = avg_conv_per_owner[avg_conv_per_owner > 50].index.tolist()
mid_perf  = avg_conv_per_owner[(avg_conv_per_owner > 20) & (avg_conv_per_owner <= 50)].index.tolist()
low_perf = avg_conv_per_owner[avg_conv_per_owner <= 1].index.tolist()

y_labels = [
    f"<b style='color:Green'>{name}</b>" if name in high_perf else
    f"<b style='color:purple'>{name}</b>" if name in mid_perf else
    f"<b style='color:Crimson'>{name}</b>" if name in low_perf else
    name
    for name in pivot_conv.index
]

fig = px.imshow(
    pivot_conv,
    text_auto='.1f',
    color_continuous_scale='matter',
    labels=dict(y="Owners", color="Conversion C1 (%)"),
    title='Conversion C1 by Owners'
)
fig.update_xaxes(side='top')
fig.update_yaxes(
    ticktext=y_labels,
    tickvals=np.arange(len(y_labels))
)
fig.update_layout(height=800, plot_bgcolor='ivory', showlegend=True)
fig.show()

In [ ]:
owners = {}
owners['avg_deals < 5'] = low_performers
owners['C1 > 50%'] = high_perf
owners['(20% < C1 <= 50%'] = mid_perf
owners['C1 < 1%'] = low_perf

In [ ]:
df_pr = df_deals[
    (df_deals['Product'] != 'Unknown') &
    (df_deals['Deal Owner Name'] != 'Unknown') &
    (df_deals['Quality'] != 'Unknown')
    ].copy()

In [ ]:
df = df_pr.copy()
df['is_success'] = (df['Stage'].astype(str).str.strip().str.lower() == 'payment done').astype(int)
agg = df.groupby('Payment Type').agg(
    total_deals=('Id','count'),
    success_count=('is_success','sum')
).reset_index()
agg['success_rate'] = agg['success_count'] / agg['total_deals']
agg = agg.sort_values('total_deals', ascending=False)
agg.head()

,Payment Type,total_deals,success_count,success_rate
3,Unknown,3031,476,0.157044
1,Recurring Payments,340,248,0.729412
0,One Payment,132,110,0.833333
2,Reservation,5,1,0.200000


In [ ]:
def consolidate_low_freq(series, min_count):
    ''' Merges low-frequency categories into the 'Others' bucket '''

    vc = series.value_counts()
    to_other = vc[vc < min_count].index
    return series.where(~series.isin(to_other), other='Other')

In [ ]:
quality_col = 'Quality'
product_col = 'Product'
min_count = 20
drop_unknown = True

df = df_deals.copy()
for c in [quality_col, product_col]:
    if c not in df.columns:
        raise KeyError(f"Колонка '{c}' не найдена в df_deals")
    df[c] = df[c].astype(str).fillna('Unknown').str.strip()

if drop_unknown:
    df = df[(df[quality_col] != 'Unknown') & (df[product_col] != 'Unknown')]

if min_count and min_count > 0:
    df[quality_col] = consolidate_low_freq(df[quality_col], min_count)
    df[product_col] = consolidate_low_freq(df[product_col], min_count)

counts = (df.groupby([quality_col, product_col])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False))

fig = px.sunburst(
    counts,
    path=[product_col, quality_col],
    values='count',
    color='count',
    color_continuous_scale="matter",
    title=f'Deals Qality by Product'
)

fig.update_traces(textinfo='label+percent entry')
fig.update_layout(margin=dict(t=40, l=10, r=10, b=10), height=500)

Hier ist die Verteilung der Produkte und die Qualität der Transaktionen gut zu erkennen. In jedem Segment ist der Anteil in Prozent angegeben, und die Farbe zeigt die Menge an. Bei jedem Produkt überwiegt ein niedriges Transaktionsniveau.

In [ ]:
df_deals.describe()

,Closing Date,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,SLA Seconds,Month,Duration days,Duration hours
count,13057,19657,3513.000000,835.000000,3220.00000,3267.000000,1.485700e+04,13057,13057.000000,13057.000000
mean,2024-01-28 11:11:48.410814208,2024-01-27 18:49:19.205372160,10.195844,5.449102,1168.97795,8964.710438,1.117224e+05,2024-01-13 02:54:54.784406784,15.341732,377.755941
min,2023-07-03 17:03:00,2022-10-11 00:00:00,6.000000,0.000000,9.00000,1200.000000,3.000000e+00,2023-07-01 00:00:00,0.000000,0.016667
25%,2023-11-11 00:00:00,2023-11-18 14:47:00,11.000000,3.000000,1000.00000,5000.000000,4.358000e+03,2023-11-01 00:00:00,0.000000,15.916667
50%,2024-02-08 00:00:00,2024-02-05 20:37:00,11.000000,5.000000,1000.00000,11000.000000,1.960300e+04,2024-02-01 00:00:00,3.000000,84.766667
75%,2024-04-18 00:00:00,2024-04-14 10:22:00,11.000000,8.000000,1000.00000,11000.000000,5.575200e+04,2024-04-01 00:00:00,12.000000,302.933333
max,2024-12-11 00:00:00,2024-06-21 15:30:00,11.000000,11.000000,11000.00000,11500.000000,2.690846e+07,2024-12-01 00:00:00,357.000000,8578.600000
std,NaN,NaN,1.837137,2.920733,1387.01313,3251.083863,7.292200e+05,NaN,32.481838,779.554256


In [ ]:
mean = df_deals['Duration days'].mean()
median = df_deals['Duration days'].median()
p95 = df_deals['Duration days'].quantile(0.95)
p99 = df_deals['Duration days'].quantile(0.99)

In [ ]:
df_clip = df_deals[df_deals['Duration days'] <= p95]

fig = px.histogram(
    df_clip,
    x='Duration days',
    nbins=50,
    title='Histogram (<=95th percentile)'
    )

fig.add_vline(
    x=mean,
    line_dash="dash",
    line_color="red",
    annotation_text=f"mean={mean:.1f}",
    annotation_position="top right")

fig.add_vline(
    x=median,
    line_dash="dot",
    line_color="green",
    annotation_text=f"median={median:.1f}",
    annotation_position="top left")

fig.update_layout(xaxis_title='Duration days', yaxis_title='count')
fig.show()

Die mittlere Dauer einer Transaktion beträgt etwa drei Tage, aber seltene langwierige Transaktionen erhöhen den Durchschnitt. Das Diagramm basiert auf 95 % der Daten ohne Ausreißer.

In [ ]:
grouped = (
    df_deals
    .groupby(["Deal Owner Name", "Month"])["Duration hours"]
    .mean()
    .reset_index(name="Average Duration (h)")
)

pivot_table = grouped.pivot_table(
    index="Deal Owner Name",
    columns="Month",
    values="Average Duration (h)",
)

fig = px.imshow(
    pivot_table,
    text_auto=".1f",
    color_continuous_scale="matter",
    labels=dict(y="Owners", color="Average Duration (h)"),
    title="Deals Average Duration by Owners",
)

fig.update_xaxes(side="top")
fig.update_layout(
    height=800,
    showlegend=True,
    plot_bgcolor='ivory'
)
fig.show()

Hier sind die zuvor erwähnten beiden Transaktionen mit 0 % Konversion zu sehen, und nun ist erkennbar, dass ihre Dauer ungewöhnlich lang ist, was bedeutet, dass sie höchstwahrscheinlich vergessen wurden zu schließen, oder dass es sich um einen Fehler beim Import der Daten handelt.

In [ ]:
grouped_sla = (
    df_deals
    .groupby(['Deal Owner Name', 'Month'])['SLA Seconds']
    .mean()
    .reset_index(name='Average SLA')
)

grouped_sla['Average SLA'] = grouped_sla['Average SLA'] / 3600

pivot_sla = grouped_sla.pivot_table(
    index='Deal Owner Name',
    columns='Month',
    values='Average SLA'
)

fig = px.imshow(
    pivot_sla,
    text_auto='.1f',
    color_continuous_scale='matter',
    labels=dict(y="Owners", color="Average SLA (h)"),
    title='Average SLA by Owners'
)

fig.update_xaxes(side='top')
fig.update_layout(height=800, plot_bgcolor='ivory', showlegend=True)
fig.show()

 Es wurden Werte von 7474,6 Stunden und 1315,4 Stunden festgestellt. Wahrscheinlich handelt es sich hierbei um einen Fehler, daher ist es sinnvoll, diese Daten aus der Analyse auszuschließen.

In [ ]:
grouped_sla = (
    df_deals
    .groupby(['Deal Owner Name', 'Month'])['SLA Seconds']
    .mean()
    .reset_index(name='Average SLA')
)

grouped_sla['Average SLA'] = grouped_sla['Average SLA'] / 3600
grouped_sla = grouped_sla[grouped_sla['Average SLA'] <= 720]

pivot_sla = grouped_sla.pivot_table(
    index='Deal Owner Name',
    columns='Month',
    values='Average SLA'
)

slow_managers = pivot_sla[pivot_sla > 150].dropna(how='all').index.tolist()
y_labels = [
    f"<b style='color:red'>{name}</b>" if name in slow_managers else name
    for name in pivot_sla.index
]

fig = px.imshow(
    pivot_sla,
    text_auto='.1f',
    color_continuous_scale='matter',
    labels=dict(y="Owners", color="Average SLA (h)"),
    title="Average SLA by Owners"
)

fig.update_xaxes(side='top')
fig.update_yaxes(
    ticktext=y_labels,
    tickvals=np.arange(len(y_labels))
)
fig.update_layout(height=800, plot_bgcolor='ivory', showlegend=True)

fig.show()

In [ ]:
owners = {}
owners['avg_deals < 5'] = low_performers
owners['C1 > 50%'] = high_perf
owners['20% < C1 <= 50%'] = mid_perf
owners['C1 < 1%'] = low_perf
owners['SLA > 100h'] = slow_managers

In [ ]:
df_outsiders = pd.DataFrame(dict([(key, pd.Series(value)) for key, value in owners.items()]))

display(df_outsiders)

,avg_deals < 5,C1 > 50%,20% < C1 <= 50%,C1 < 1%,SLA > 100h
0,Alice Johnson,John Doe,George King,Alice Johnson,Cara Iverson
1,Amy Green,NaN,Oliver Taylor,Amy Green,Ian Miller
2,George King,NaN,NaN,Bob Brown,Julia Nelson
3,John Doe,NaN,NaN,Cara Iverson,Kevin Parker
4,Sam Young,NaN,NaN,Diana Evans,Oliver Taylor
5,Wendy Clark,NaN,NaN,Rachel White,Quincy Vincent
6,Xander Dean,NaN,NaN,Sam Young,Rachel White
7,Yara Edwards,NaN,NaN,Wendy Clark,NaN
8,NaN,NaN,NaN,Xander Dean,NaN
9,NaN,NaN,NaN,Yara Edwards,NaN


In [ ]:
df_all_managers = pd.DataFrame({'Manager': df_deals['Deal Owner Name'].dropna().unique()})
df_all_managers = df_all_managers[df_all_managers['Manager'] != 'Unknown'].copy()

df_outsiders_long = df_outsiders.melt(var_name='Reason', value_name='Manager').dropna(subset=['Manager'])

outsiders_counts = df_outsiders_long['Manager'].value_counts().reset_index()
outsiders_counts.columns = ['Manager', 'Res_cnt']

summary = pd.merge(df_all_managers, outsiders_counts, on='Manager', how='left')
summary['Res_cnt'] = summary['Res_cnt'].fillna(0).astype(int)

summary['Status'] = summary['Res_cnt'].apply(
    lambda x: 'Стабильный результат' if x == 0 else 'Требует внимания'
)

reasons = df_outsiders_long.groupby('Manager')['Reason'].apply(lambda x: ', '.join(set(x))).reset_index()
summary = summary.merge(reasons, on='Manager', how='left')

def highlight_rows(row):
    if row['Status'] == 'Требует внимания':
        return ['background-color: #ffe0e0; color: #b30000; font-weight: bold'] * len(row)
    else:
        return ['background-color: #e0ffe0; color: #006600; font-weight: bold'] * len(row)

styled_summary = summary.style.apply(highlight_rows, axis=1)
display(styled_summary)

,Manager,Res_cnt,Status,Reason
0,Ben Hall,0,Стабильный результат,nan
1,Ulysses Adams,0,Стабильный результат,nan
2,Eva Kent,0,Стабильный результат,nan
3,Paula Underwood,0,Стабильный результат,nan
4,Victor Barnes,0,Стабильный результат,nan
5,Amy Green,2,Требует внимания,"C1 < 1%, avg_deals < 5"
6,Ian Miller,1,Требует внимания,SLA > 100h
7,Rachel White,2,Требует внимания,"C1 < 1%, SLA > 100h"
8,Quincy Vincent,1,Требует внимания,SLA > 100h
9,Charlie Davis,0,Стабильный результат,nan


In [ ]:
status_map = {
    "Стабильный результат": "Stable",
    "Требует внимания": "At Risk"
}

summary['Owner Status'] = summary['Status'].map(status_map)

df_deals = df_deals.merge(
    summary[['Manager', 'Owner Status']],
    left_on='Deal Owner Name',
    right_on='Manager',
    how='left'
)

df_deals.drop(columns=['Manager'], inplace=True)

In [ ]:
display(summary[['Manager', 'Owner Status']].head())

,Manager,Owner Status
0,Ben Hall,Stable
1,Ulysses Adams,Stable
2,Eva Kent,Stable
3,Paula Underwood,Stable
4,Victor Barnes,Stable


In [ ]:
df_deals['Owner Status'].value_counts()

,count
Owner Status,
Stable,11884
At Risk,7744


In [ ]:
df_deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,SLA Seconds,Deutsch_Level_Clean,Deal Date,Deal Time,Close Date,Close Time,Month,Duration days,Duration hours,Owner Status
0,5805028000056864695,Ben Hall,NaT,Unknown,New Lead,Unknown,/eng/test,03.07.23women,v16,women,...,NaN,Unknown,2024-06-21,15:30:00,NaT,NaT,NaT,NaN,NaN,NaN
1,5805028000056859489,Ulysses Adams,NaT,Unknown,New Lead,Unknown,/at-eng,Unknown,Unknown,Unknown,...,NaN,Unknown,2024-06-21,15:23:00,NaT,NaT,NaT,NaN,NaN,NaN
2,5805028000056832357,Ulysses Adams,2024-06-21 14:45:00,D - Non Target,Lost,Non target,/at-eng,engwien_AT,b1-at,21_06_2024,...,1603.0,Unknown,2024-06-21,00:00:00,2024-06-21,14:45:00,2024-06-01,0.0,14.750000,NaN
3,5805028000056824246,Eva Kent,2024-06-21 13:32:00,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,bloggersvideo14com,recentlymoved,...,3604.0,Unknown,2024-06-21,00:00:00,2024-06-21,13:32:00,2024-06-01,0.0,13.533333,NaN
4,5805028000056873292,Ben Hall,2024-06-21 13:21:00,D - Non Target,Lost,Non target,/eng,discovery_DE,website,Unknown,...,3192.0,Unknown,2024-06-21,00:00:00,2024-06-21,13:21:00,2024-06-01,0.0,13.350000,NaN


In [ ]:
df_calls['Call Status'].value_counts()

,count
Call Status,
Attended Dialled,69522
Unattended Dialled,14146
Missed,5743
Received,3070
Overdue,57
Scheduled Attended Delay,20
Cancelled,19
Scheduled Unattended Delay,17
Scheduled Attended,14


In [ ]:
top_statuses = df_calls['Call Status'].value_counts().nlargest(4).index
df_top = df_calls[df_calls['Call Status'].isin(top_statuses)]

risky_owners = df_deals.loc[df_deals['Owner Status'] == 'At Risk', 'Deal Owner Name']

df_grouped = (
    df_top
    .groupby(['Call Owner Name', 'Call Status'])
    .size()
    .reset_index(name='Calls Count')
)
df_grouped = df_grouped[
    (~df_grouped['Call Owner Name'].isin(risky_owners)) &
    (df_grouped['Call Owner Name']!= 'Tina Zhang') &
    (df_grouped['Call Owner Name']!= 'Laura Quinn')
]

fig = px.bar(
    df_grouped,
    x='Call Status',
    y='Calls Count',
    color='Call Status',
    facet_col='Call Owner Name',
    facet_col_wrap=7,
    facet_row_spacing=0.12,
    facet_col_spacing=0.03,
    title='Average Calls and Call Status for Not-Risk Owners',
    color_discrete_sequence=px.colors.qualitative.Plotly
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_traces(marker=dict(opacity=1))
fig.update_layout(
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=700,
    margin=dict(t=100, l=50, r=50, b=50)
)
fig.update_xaxes(title='', tickangle=45)
fig.show()


Das ermöglicht es, die Qualität der Kommunikation mit potenziellen Kunden zu sehen und zu analysieren. Auf dieser Grundlage baue ich meine Hypothesen für den Bereich Produktanalyse auf.

#Analyse der Kampagneneffizienz:
1. Vergleichen Sie die Effizienz verschiedener Kampagnen hinsichtlich Leadgenerierung
und Konversionsrate.
2. Bewerten Sie die Effektivität verschiedener Marketingquellen Source bei der
Generierung qualitativ hochwertiger Leads.

##1. Vergleichen Sie die Effizienz verschiedener Kampagnen hinsichtlich Leadgenerierung und Konversionsrate.

In [ ]:
df_deals.head(1)

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,SLA Seconds,Deutsch_Level_Clean,Deal Date,Deal Time,Close Date,Close Time,Month,Duration days,Duration hours,Owner Status
0,5805028000056864695,Ben Hall,NaT,Unknown,New Lead,Unknown,/eng/test,03.07.23women,v16,women,...,NaN,Unknown,2024-06-21,15:30:00,NaT,NaT,NaT,NaN,NaN,NaN


In [ ]:
top_n = 10

df = df_deals[df_deals['Campaign'] != 'Unknown']

df['is_success'] = (df['Stage'].astype(str).str.strip().str.lower() == 'payment done').astype(int)

agg = (df.groupby('Campaign')
         .agg(total_deals=('Id', 'count'),
              success_count=('is_success', 'sum'))
         .reset_index())

agg['failed_count'] = agg['total_deals'] - agg['success_count']
agg['conversion'] = (agg['success_count'] / agg['total_deals']).fillna(0)

agg_by_volume = agg.sort_values('total_deals', ascending=False).reset_index(drop=True)
top = agg_by_volume.head(top_n).copy()

/tmp/ipython-input-3372987290.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
top

,Campaign,total_deals,success_count,failed_count,conversion
0,performancemax_digitalmarkt_ru_DE,2573,112,2461,0.043529
1,youtube_shorts_DE,1596,53,1543,0.033208
2,12.07.2023wide_DE,1531,48,1483,0.031352
3,02.07.23wide_DE,943,52,891,0.055143
4,04.07.23recentlymoved_DE,734,31,703,0.042234
5,03.07.23women,595,31,564,0.052101
6,Dis_DE,567,30,537,0.052910
7,07.07.23LAL_DE,529,28,501,0.052930
8,12.09.23interests_Uxui_DE,515,27,488,0.052427
9,24.09.23retargeting_DE,472,17,455,0.036017


In [ ]:
fig = px.scatter(
    top, x='total_deals',
    y='conversion',
    size='success_count',
    color='Campaign',
    title='Volume vs Conversion by Campaign',
    labels={'conversion':'Conversion rate'},
    size_max=50
    )

fig.update_yaxes(tickformat='.0%')
fig.add_hline(
    y=top['conversion'].median(),
    line_dash='dash',
    line_color='gray')

fig.update_layout(plot_bgcolor='ivory')
fig.show()

Bei gleichem Lead-Volumen variiert die Effizienz stark. Einige Unternehmen haben einen hohen Traffic, aber eine geringe Konversionsrate, andere hingegen das Gegenteil.

##2. Bewerten Sie die Effektivität verschiedener Marketingquellen 'Source' bei der Generierung qualitativ hochwertiger Leads.

In [ ]:
top_n = 10

df = df_deals[df_deals['Source'] != 'Unknown']

df['is_success'] = (df['Stage'].astype(str).str.strip().str.lower() == 'payment done').astype(int)

agg = (df.groupby('Source') #
         .agg(total_deals=('Id', 'count'),
              success_count=('is_success', 'sum'))
         .reset_index())

agg['failed_count'] = agg['total_deals'] - agg['success_count']
agg['conversion'] = (agg['success_count'] / agg['total_deals']).fillna(0)

agg_by_volume = agg.sort_values('total_deals', ascending=False).reset_index(drop=True)
top = agg_by_volume.head(top_n).copy()

In [ ]:
fig = px.scatter(
    top,
    x='total_deals',
    y='conversion',
    size='success_count',
    color='Source',
    title='Volume vs Conversion by Source',
    labels={'conversion':'Conversion rate'},
    size_max=30
    )

fig.update_yaxes(tickformat='.0%')
fig.add_hline(
    y=top['conversion'].median(),
    line_dash='dash',
    line_color='gray'
    )

fig.update_layout(plot_bgcolor='ivory')
fig.show()

Es ist ersichtlich, dass Quellen wie Organic, SMM, Webinar und Telegram eine recht gute Konversion erzielen. Facebook und Google sind eher auf die Reichweite des Publikums ausgerichtet.

#Analyse der Vertriebseffizienz:
1. Bewerten Sie die Effizienz einzelner Deal-Owner und Werbekampagnen in Bezug auf
die Anzahl der bearbeiteten Deals, Konversionsraten und den Gesamtumsatz.

##1. Bewerten Sie die Effizienz einzelner Deal-Owner und Werbekampagnen in Bezug auf die Anzahl der bearbeiteten Deals, Konversionsraten und den Gesamtumsatz.

In [ ]:
deal_owner_agg = (df_deals
    .groupby("Deal Owner Name")
    .agg(total_deals=('Id', 'count'),
         success_count=('Stage', lambda x: (x.astype(str).str.strip().str.lower() == 'payment done').sum()),
         total_amount=('Offer Total Amount', lambda x: x.sum() if x.dtype == 'float64' else 0),
         average_sla=('SLA Seconds', lambda x: x.mean() / 3600 if x.dtype == 'float64' else None))
    .reset_index())

deal_owner_agg['failed_count'] = deal_owner_agg['total_deals'] - deal_owner_agg['success_count']
deal_owner_agg['conversion'] = (deal_owner_agg['success_count'] / deal_owner_agg['total_deals']).fillna(0)
deal_owner_agg['avg_amount_per_deal'] = (deal_owner_agg['total_amount'] / deal_owner_agg['total_deals']).fillna(0)

deal_owner_agg = deal_owner_agg[deal_owner_agg['total_deals'] > 0].copy()

In [ ]:
min_deals = 5

good = deal_owner_agg[deal_owner_agg['total_deals'] >= min_deals].copy()

fig = px.scatter(
    good,
    x='total_deals',
    y='conversion',
    size='success_count',
    color='Deal Owner Name',
    title=f'Volume vs Conversion by Deal Owner (only >= {min_deals} deals)',
    labels={'conversion':'Conversion rate'},
    size_max=50
    )

fig.update_yaxes(tickformat='.0%')
fig.add_hline(
    y=good['conversion'].median(),
    line_dash='dash',
    line_color='gray',
    annotation_text='median conv',
    annotation_position='top left'
    )

fig.update_layout(plot_bgcolor='ivory')
fig.show()

Ich habe die Arbeit aller Manager oben ausführlicher beschrieben, hier kann man sich einen Überblick verschaffen. Auf diesem interaktiven Diagramm kann man einige Manager aus meiner „roten Liste” ausschalten, dann wird das Bild klarer und die eindeutigen Spitzenreiter werden sichtbar.

In [ ]:
min_deals = 5
good = deal_owner_agg[deal_owner_agg['total_amount'] >= min_deals].copy()

fig = px.scatter(
    good,
    x='total_amount',
    y='conversion',
    size='success_count',
    color='Deal Owner Name',
    title=f'Amount vs Conversion by Deal Owner (only >= {min_deals} deals)',
    labels={'conversion':'Conversion rate', 'total_amount': 'Total Amount'},
    size_max=50
    )

fig.update_yaxes(tickformat='.0%')
fig.add_hline(
    y=good['conversion'].median(),
    line_dash='dash',
    line_color='gray',
    annotation_text='median conv',
    annotation_position='top left'
    )

fig.update_layout(plot_bgcolor='ivory')
fig.show()

#Analyse von Zahlungen und Produkten:
1. Untersuchen Sie die Verteilung der Zahlungsarten und deren Einfluss auf den Erfolg
von Deals.
2. Analysieren Sie die Beliebtheit und den Erfolg verschiedener Produkte und
Lernformate.

##1. Untersuchen Sie die Verteilung der Zahlungsarten und deren Einfluss auf den Erfolg von Deals.

In [ ]:
df= df_deals[
    (df_deals['Product'] != 'Unknown') &
    (df_deals['Education Type'] != 'Unknown') &
    (df_deals['Payment Type'] != 'Unknown')
    ].copy()

In [ ]:
df['Payment Type'] = df['Payment Type'].astype(str).str.strip().fillna('Unknown')
df['is_success'] = (df['Stage'].astype(str).str.strip().str.lower() == 'payment done').astype(int)
agg = df.groupby('Payment Type').agg(
    total_deals=('Id','count'),
    success_count=('is_success','sum')
).reset_index()
agg['success_rate'] = agg['success_count'] / agg['total_deals']
agg = agg.sort_values('total_deals', ascending=False)
agg.head()

,Payment Type,total_deals,success_count,success_rate
1,Recurring Payments,334,247,0.739521
0,One Payment,131,110,0.839695
2,Reservation,5,1,0.200000


In [ ]:
def consolidate_low_freq(series, min_count):
    vc = series.value_counts()
    to_other = vc[vc < min_count].index
    return series.where(~series.isin(to_other), other='Other')

In [ ]:
product_col = 'Product'
payment_col = 'Payment Type'
stage_col = 'Stage'
min_count = 20

if min_count and min_count > 0:
    df[payment_col] = consolidate_low_freq(df[payment_col], min_count)
    df[stage_col] = consolidate_low_freq(df[stage_col], min_count)

counts = (df.groupby([product_col, payment_col, stage_col])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False))

fig = px.sunburst(
    counts,
    path=[product_col, payment_col, stage_col],
    values='count',
    color='count',
    color_continuous_scale="matter",
    title=f'Sunburst: {product_col} → {payment_col} → {stage_col} (count)'
)

fig.update_traces(textinfo='label+percent entry')
fig.update_layout(margin=dict(t=40, l=10, r=10, b=10), height=600)
fig.show()

Hier ist die Verteilung der Produkte, Zahlungsarten und Zahlungsmethoden gut zu sehen. In jedem Segment ist der Anteil in Prozent angegeben, und die Farbe zeigt die Menge an. Außerdem können Sie dank der Interaktivität filtern und die für Sie interessanten Kennzahlen genauer betrachten.

##2. Analysieren Sie die Beliebtheit und den Erfolg verschiedener Produkte und Lernformate.

In [ ]:
agg = df.groupby(['Month', 'Product']).agg(
    deals_count=('Id', 'count'),
    success_count=('Stage', lambda s: (s.astype(str).str.strip().str.lower() == 'payment done').sum())
).reset_index()

agg_long = agg.melt(id_vars=['Month', 'Product'], value_vars=['deals_count', 'success_count'],
                    var_name='metric', value_name='count')
agg_long['metric'] = agg_long['metric'].map({'deals_count': 'All deals', 'success_count': 'Payment Done'})

fig = px.line(
    agg_long,
    x='Month',
    y='count',
    color='metric',
    facet_col='Product',
    facet_col_wrap=3,
    markers=True,
    title='Monthly deals by Product: total deals vs Payment Done'
)

fig.update_layout(height=600, legend_title_text='')
fig.update_xaxes(tickformat='%b %Y', tickangle=45)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

In [ ]:
agg = df.groupby(['Month', 'Education Type']).agg(
    deals_count=('Id', 'count'),
    success_count=('Stage', lambda s: (s.astype(str).str.strip().str.lower() == 'payment done').sum())
).reset_index()

agg = agg[agg['Education Type'] != 'Unknown'].copy()

agg_long = agg.melt(id_vars=['Month', 'Education Type'], value_vars=['deals_count', 'success_count'],
                    var_name='metric', value_name='count')
agg_long['metric'] = agg_long['metric'].map({'deals_count': 'All deals', 'success_count': 'Payment Done'})

fig = px.line(
    agg_long,
    x='Month',
    y='count',
    color='metric',
    facet_col='Education Type',
    facet_col_wrap=3,
    markers=True,
    title='Monthly deals by Education Type (excluding Unknown): All deals vs Payment Done'
)

fig.update_layout(height=600, legend_title_text='')
fig.update_xaxes(tickformat='%b %Y', tickangle=45)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()

In [ ]:
product_col = 'Product'
education_col = 'Education Type'
stage_col = 'Stage'
min_count = 20

if min_count and min_count > 0:
    df[education_col] = consolidate_low_freq(df[education_col], min_count)
    df[stage_col] = consolidate_low_freq(df[stage_col], min_count)

counts = (df.groupby([product_col, education_col, stage_col])
            .size()
            .reset_index(name='count')
            .sort_values('count', ascending=False))

fig = px.sunburst(
    counts,
    path=[product_col, education_col, stage_col],
    values='count',
    color='count',
    color_continuous_scale="matter",
    title=f'Sunburst: {product_col} → {education_col} → {stage_col} (count)'
)

fig.update_traces(textinfo='label+percent entry')
fig.update_layout(margin=dict(t=40, l=10, r=10, b=10), height=600)
fig.show()

#Geografische Analyse:
1. Analysieren Sie die geografische Verteilung der Deals nach Städten.
2. Untersuchen Sie den Einfluss des Deutschkenntnisniveaus auf den Erfolg von Deals in
verschiedenen Städten.

##1. Analysieren Sie die geografische Verteilung der Deals nach Städten.

In [ ]:
import json

with open("city_data_google_en.json", "r") as json_file:
    city_data = json.load(json_file)

In [ ]:
def get_city_info(city):
    info = city_data.get(city, {})
    return pd.Series({
        'longitude': info.get('longitude', None),
        'latitude': info.get('latitude', None),
        'country': info.get('country', None),
        'city': info.get('city', None),
        'federal_state': info.get('federal_state', None)
    })

df_deals['City'] = df_deals['City'].fillna('Unknown')
df_deals[['longitude', 'latitude', 'country', 'city', 'federal_state']] = df_deals['City'].apply(get_city_info)

In [ ]:
df_deals.to_csv("deals_geo.csv", index=False)

In [ ]:
df_deals.loc[df_deals.City != "Unknown"].head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Duration days,Duration hours,Owner Status_x,Owner Status_y,Owner Status,longitude,latitude,country,city,federal_state
32,5805028000056714532,Ulysses Adams,NaT,Unknown,Registered on Webinar,Unknown,/webinar,webinar1906,Unknown,invitation,...,NaN,NaN,NaN,Stable,Stable,13.404954,52.520007,Germany,Berlin,Berlin
39,5805028000056731279,Ulysses Adams,NaT,Unknown,Registered on Webinar,Unknown,/webinar,Unknown,Unknown,Unknown,...,NaN,NaN,NaN,Stable,Stable,7.609363,50.307827,Germany,Lahnstein,Rhineland-Palatinate
49,5805028000056683030,Charlie Davis,NaT,C - Low,Waiting For Payment,Unknown,/eng/test,performancemax_eng_DE,_{region_name}_,Unknown,...,NaN,NaN,NaN,Stable,Stable,10.063357,49.133735,Germany,Crailsheim,Baden-Württemberg
50,5805028000056568397,Paula Underwood,NaT,Unknown,Registered on Webinar,Unknown,/webinar,Unknown,Unknown,Unknown,...,NaN,NaN,NaN,Stable,Stable,13.862376,53.316844,Germany,Prenzlau,Brandenburg
53,5805028000056558351,Ulysses Adams,NaT,C - Low,Waiting For Payment,Unknown,/eng,Unknown,Unknown,Unknown,...,NaN,NaN,NaN,Stable,Stable,7.465298,51.513587,Germany,Dortmund,North Rhine-Westphalia


In [ ]:
df_deals['City'] = df_deals.City.replace('-', "Unknown")

In [ ]:
geo_deals_df = df_deals[['Id', 'City']]
geo_deals_df_clean = geo_deals_df.dropna(subset=['City'])
deals_by_city = geo_deals_df_clean.groupby('City').size().reset_index(name='Number of Deals')

deals_by_city_sorted = deals_by_city.sort_values(by='Number of Deals', ascending=False)
deals_by_city_sorted

,City,Number of Deals
778,Unknown,16983
88,Berlin,263
523,München,87
289,Hamburg,78
572,Nürnberg,52
...,...,...
12,Ajman,1
11,Aidlingen,1
10,Aichwald,1
7,Ahaus,1


In [ ]:
geo_data_deals = df_deals[["City", "longitude", "latitude", "country",
                           "city"]].drop_duplicates(subset=["City"]).dropna()

In [ ]:
deals_by_city_sorted = deals_by_city_sorted.merge(geo_data_deals, on='City', how="inner")
deals_by_city_sorted

,City,Number of Deals,longitude,latitude,country,city
0,Berlin,263,13.404954,52.520007,Germany,Berlin
1,München,87,11.581981,48.135125,Germany,München
2,Hamburg,78,9.987170,53.548828,Germany,Hamburg
3,Nürnberg,52,11.074564,49.454288,Germany,Nürnberg
4,Leipzig,52,12.373075,51.339695,Germany,Leipzig
...,...,...,...,...,...,...
862,Ajman,1,55.513643,25.405217,United Arab Emirates,Ajman
863,Aidlingen,1,8.895746,48.679056,Germany,Aidlingen
864,Aichwald,1,9.388987,48.761761,Germany,Aichwald
865,Ahaus,1,7.015006,52.082532,Germany,Ahaus


In [ ]:
fig = px.scatter_geo(deals_by_city_sorted,
                     lat='latitude',
                     lon='longitude',
                     size='Number of Deals',
                     hover_name='city',
                     projection='natural earth',
                     title="Geographic Distribution of Deals by City")
fig.show()

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

deals_df_geo = df_deals[df_deals['City'] != 'Unknown'].dropna(subset="city")

geometry = [Point(xy) for xy in zip(deals_df_geo["longitude"], deals_df_geo["latitude"])]

gdf = gpd.GeoDataFrame(deals_df_geo, geometry=geometry)
gdf.set_crs(epsg=4326, inplace=True)

display(gdf.head())

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,Content,Term,...,Duration hours,Owner Status_x,Owner Status_y,Owner Status,longitude,latitude,country,city,federal_state,geometry
32,5805028000056714532,Ulysses Adams,NaT,Unknown,Registered on Webinar,Unknown,/webinar,webinar1906,Unknown,invitation,...,NaN,NaN,Stable,Stable,13.404954,52.520007,Germany,Berlin,Berlin,POINT (13.40495 52.52001)
39,5805028000056731279,Ulysses Adams,NaT,Unknown,Registered on Webinar,Unknown,/webinar,Unknown,Unknown,Unknown,...,NaN,NaN,Stable,Stable,7.609363,50.307827,Germany,Lahnstein,Rhineland-Palatinate,POINT (7.60936 50.30783)
49,5805028000056683030,Charlie Davis,NaT,C - Low,Waiting For Payment,Unknown,/eng/test,performancemax_eng_DE,_{region_name}_,Unknown,...,NaN,NaN,Stable,Stable,10.063357,49.133735,Germany,Crailsheim,Baden-Württemberg,POINT (10.06336 49.13374)
50,5805028000056568397,Paula Underwood,NaT,Unknown,Registered on Webinar,Unknown,/webinar,Unknown,Unknown,Unknown,...,NaN,NaN,Stable,Stable,13.862376,53.316844,Germany,Prenzlau,Brandenburg,POINT (13.86238 53.31684)
53,5805028000056558351,Ulysses Adams,NaT,C - Low,Waiting For Payment,Unknown,/eng,Unknown,Unknown,Unknown,...,NaN,NaN,Stable,Stable,7.465298,51.513587,Germany,Dortmund,North Rhine-Westphalia,POINT (7.4653 51.51359)


In [ ]:
import folium

center = gdf.geometry.union_all().centroid
m = folium.Map(location=[center.y, center.x], zoom_start=4)

for idx, row in gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="red",
        fill=True,
        fill_color="red",
        popup=f"Index: {idx}"
    ).add_to(m)

m.save("deals_map.html")
m

In [ ]:
city_analysis = df_deals[df_deals["city"] != "Unknown"].groupby('city').agg(
    Total_Deals=('Stage', 'count'),
    Successful_Deals=('Stage', lambda x: (x == 'Payment Done').sum()),
    Conversion_Rate=('Stage', lambda x: (x == 'Payment Done').sum() / len(x) * 100 if len(x) > 0 else 0)
).reset_index()

fig2 = px.scatter(
    city_analysis,
    x='city',
    y='Total_Deals',
    size='Successful_Deals',
    color='Conversion_Rate',
    hover_name='city',
    title='Total Deals by City',
    labels={
        'Total_Deals': 'Total Deals',
        'Conversion_Rate': 'Conversion Rate (%)'
    },
    color_continuous_scale='matter',
    size_max=30
)
fig2.show()

##2. Untersuchen Sie den Einfluss des Deutschkenntnisniveaus auf den Erfolg von Deals in verschiedenen Städten.

In [ ]:
deutsch_level_analysis = df_deals[df_deals["Deutsch_Level_Clean"] != "Unknown"].groupby('Deutsch_Level_Clean').agg(
    Total_Deals=('Stage', 'count'),
    Successful_Deals=('Stage', lambda x: (x == 'Payment Done').sum()),
    Conversion_Rate=('Stage', lambda x: (x == 'Payment Done').sum() / len(x) * 100 if len(x) > 0 else 0)
).reset_index()


fig3 = px.scatter(
    deutsch_level_analysis,
    x='Deutsch_Level_Clean',
    y='Total_Deals',
    size='Successful_Deals',
    color='Conversion_Rate',
    hover_name='Deutsch_Level_Clean',
    title='Total Deals by Level of Deutsch',
    labels={
        'Total_Deals': 'Total Deals',
        'Conversion_Rate': 'Conversion Rate (%)'
    },
    color_continuous_scale='matter',
    size_max=30
)
fig3.show()

In [ ]:
df_deals.to_pickle('2_Deals.pkl')